In [1]:
from ugropy import unifac

import pandas as pd

In [ ]:
df = pd.read_csv("ddbst_tests.csv", index_col=1, skiprows=3)

df

,code,number,solution
smiles,,,
CC1=C(C(=CC(=C1Cl)Cl)Cl)Cl,ZOXPZFFPPKVNEA-UHFFFAOYSA-N,13870,9:1|11:1|53:4
CCC=CCC1=C(CCC1=O)C,XMLSXPIVAXONDL-UHFFFAOYSA-N,10261,1:2|2:3|19:1|70:1|6:1
C#CCCl,LJZPPWWHKPGCHS-UHFFFAOYSA-N,12221,65:1|44:1
C1=CN=CC=C1CCN,IDLHTECVNDEOIY-UHFFFAOYSA-N,83275,2:1|29:1|38:1
C[Si](C)(C)C[Si](C)(C)CCl,WBZTYASEAQTKBK-UHFFFAOYSA-N,87574,1:5|2:1|44:1|81:2
...,...,...,...
CN(C)C1=CC=CC2=C1C=CC=C2N(C)C,TYMBOHQMUQAOBA-UHFFFAOYSA-N,139088,1:2|10:4|34:2|9:6
C=CCCCCCCCCCC=C,BPHFKBMQSYYNGQ-UHFFFAOYSA-N,30874,2:9|5:2
C=CC#N.C1=CC(=CC(=C1)CN)CN,XMEXUJUMKRUUNG-UHFFFAOYSA-N,71311341,9:4|10:2|68:1|29:2


# Auxiliar function to convert the DDBST solution string to a dictionary of groups and their counts.

In [24]:
# Function to convert the solution string into a dictionary of group 
# occurrences. E.g. 9:1|11:1|53:4 -> {'ACH': 1, 'ACCH3': 1, 'ACCL': 4}
subgroup_lookup = (
    unifac.subgroups_info
    .reset_index()
    .set_index("subgroup_number")["group"]
    .to_dict()
)

def get_sol_dict(solution: str):
    return {
        subgroup_lookup[int(g)]: int(o)
        for g, o in (group.split(":") for group in solution.split("|"))
    }

# Correct at first

In [25]:
count_total = 0                 # Total number of solutions compared
count_first_correct = 0         # Number of times the first solution matches
count_skipped = 0               # Number of solutions skipped due to "??" in the code
smiles_incorrects = []          # List of SMILES for which the first solution is incorrect
sol_expected_incorrects = []    # List of expected solutions for the incorrect cases

for smiles, code, solution in zip(df.index, df["code"], df["solution"]):

    count_total += 1

    if "??" in code:
        count_skipped += 1
        continue

    ddbst_sol = get_sol_dict(solution)
    ugropy_sol = unifac.get_groups(smiles, "smiles").subgroups

    if ddbst_sol == ugropy_sol:
        count_first_correct += 1
    else:
        smiles_incorrects.append(smiles)
        sol_expected_incorrects.append(ddbst_sol)

In [26]:
count_compared = count_total - count_skipped
count_incorrect = len(smiles_incorrects)
pct_correct = count_first_correct / count_total * 100
pct_incorrect = count_incorrect / count_total * 100

print("=" * 45)
print(f"{'UGROPY VALIDATION RESULTS':^45}")
print("=" * 45)
print(f"  Total molecules:     {count_total:>6}")
print(f"  Skipped (??):        {count_skipped:>6}")
print(f"  Compared:            {count_compared:>6}")
print("-" * 45)
print(f"  ✅ Correct:          {count_first_correct:>6}  ({pct_correct:.3f}%)")
print(f"  ❌ Incorrect:        {count_incorrect:>6}  ({pct_incorrect:.3f}%)")
print("=" * 45)

          UGROPY VALIDATION RESULTS          
  Total molecules:      28617
  Skipped (??):            29
  Compared:             28588
---------------------------------------------
  ✅ Correct:           26510  (92.637%)
  ❌ Incorrect:          2078  (7.261%)


# Additional correct multiplicity

In [ ]:
count_multiplicity_correct = 0      # Counts correct solutions for molecules with solutions multiplicity
store_multiple_solutions = []       # List multiple solutions for filters testing
smiles_multiple_correct = []        # Store SMILES of molecules with multiple correct solutions
expected_multiple_correct = []      # Store expected solutions of molecules with multiple correct solutions
incorrects_smiles = []              # Store SMILES of definitely incorrect solutions
incorrects_expected = []            # Store expected solutions of definitely incorrect solutions

for smiles, expected in zip(smiles_incorrects, sol_expected_incorrects):
    solutions = unifac.get_groups(smiles, "smiles", search_multiple_solutions=True)
    sols_groups = [sol.subgroups for sol in solutions]
    
    if expected in sols_groups:
        count_multiplicity_correct += 1
        smiles_multiple_correct.append(smiles)
        expected_multiple_correct.append(expected)
        store_multiple_solutions.append(solutions)
    else:
        incorrects_smiles.append(smiles)
        incorrects_expected.append(expected)

In [28]:
count_total_correct = count_first_correct + count_multiplicity_correct
count_definitely_incorrect = len(incorrects_smiles)
count_not_correct = count_definitely_incorrect + count_skipped
pct_total_correct = count_total_correct / count_total * 100
pct_multi = count_multiplicity_correct / count_total * 100
pct_not_correct = count_not_correct / count_total * 100

print("=" * 45)
print(f"{'UGROPY VALIDATION RESULTS':^45}")
print("=" * 45)
print(f"  Total molecules:     {count_total:>6}")
print(f"  Skipped (??):        {count_skipped:>6}")
print(f"  Compared:            {count_compared:>6}")
print("-" * 45)
print(f"  ✅ Correct (first):  {count_first_correct:>6}  ({pct_correct:.3f}%)")
print(f"  ✅ Correct (multiple):  {count_multiplicity_correct:>6}  ({pct_multi:.3f}%)")
print(f"                      {'─'*16}")
print(f"  ✅ Total correct:    {count_total_correct:>6}  ({pct_total_correct:.3f}%)")
print(f"  ❌ Incorrect:        {count_not_correct:>6}  ({pct_not_correct:.3f}%)")
print("=" * 45)

          UGROPY VALIDATION RESULTS          
  Total molecules:      28617
  Skipped (??):            29
  Compared:             28588
---------------------------------------------
  ✅ Correct (first):   26510  (92.637%)
  ✅ Correct (multiple):    1816  (6.346%)
                      ────────────────
  ✅ Total correct:     28326  (98.983%)
  ❌ Incorrect:           291  (1.017%)
